Making Dynamic Image

Dynamic Image Function

In [ ]:
def get_dynamic_image(frames, normalized=True):
    """ Takes a list of frames and returns either a raw or normalized dynamic image."""
    num_channels = frames[0].shape[2]
    channel_frames = _get_channel_frames(frames, num_channels)
    channel_dynamic_images = [_compute_dynamic_image(channel) for channel in channel_frames]

    dynamic_image = cv2.merge(tuple(channel_dynamic_images))
    if normalized:
        dynamic_image = cv2.normalize(dynamic_image, None, 0, 255, norm_type=cv2.NORM_MINMAX)
        dynamic_image = dynamic_image.astype('uint8')

    return dynamic_image


def _get_channel_frames(iter_frames, num_channels):
    """ Takes a list of frames and returns a list of frame lists split by channel. """
    frames = [[] for channel in range(num_channels)]

    for frame in iter_frames:
        for channel_frames, channel in zip(frames, cv2.split(frame)):
            channel_frames.append(channel.reshape((*channel.shape[0:2], 1)))
    for i in range(len(frames)):
        frames[i] = np.array(frames[i])
    return frames

def _compute_dynamic_image(frames):
    """ Adapted from https://github.com/hbilen/dynamic-image-nets """
    num_frames, h, w, depth = frames.shape

    # Compute the coefficients for the frames.
    coefficients = np.zeros(num_frames)
    for n in range(num_frames):
        cumulative_indices = np.array(range(n, num_frames)) + 1
        coefficients[n] = np.sum(((2*cumulative_indices) - num_frames) / cumulative_indices)

    # Multiply by the frames by the coefficients and sum the result.
    x1 = np.expand_dims(frames, axis=0)
    x2 = np.reshape(coefficients, (num_frames, 1, 1, 1))
    result = x1 * x2
    return np.sum(result[0], axis=0).squeeze()

#Reading Frames from Video
def load_frames_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Resizing Image
        frame = cv2.resize(frame, (224, 224))
        frames.append(frame)
    cap.release()
    return frames

Generate Dynamic Image

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm


# ===============================
# FUNCTION: Load Frames
# ===============================
def load_frames_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame.astype(np.float32))

    cap.release()
    return frames


# ===============================
# FUNCTION: Dynamic Image
# ===============================
def get_dynamic_image(frames, normalized=True):
    num_frames = len(frames)

    coefficients = np.array(
        [2 * (i + 1) - num_frames - 1 for i in range(num_frames)],
        dtype=np.float32
    )

    dynamic_image = np.zeros_like(frames[0])

    for i in range(num_frames):
        dynamic_image += coefficients[i] * frames[i]

    if normalized:
        dynamic_image = cv2.normalize(dynamic_image, None, 0, 255, cv2.NORM_MINMAX)

    return dynamic_image.astype(np.uint8)


# ===============================
# MAIN FUNCTION
# ===============================
def main():
    input_root = "DATASET_CROPPED_NEW_ADDED"
    output_root = "dataset_final"

    if not os.path.exists(input_root):
        print(f"Error: Folder '{input_root}' tidak ditemukan.")
        return

    print(f"Memulai pembuatan Dynamic Image dari '{input_root}'...")
    print(f"Hasil akan disimpan di '{output_root}'\n")

    # Loop through all classes
    for class_name in os.listdir(input_root):
        class_path = os.path.join(input_root, class_name)
        # if class_name.lower() != "friend":
            # continue
        if not os.path.isdir(class_path):
            continue

        # Buat folder output class
        output_class_path = os.path.join(output_root, class_name)
        os.makedirs(output_class_path, exist_ok=True)

        print(f"--- Memproses Kelas: {class_name} ---")

        # 🔹 Loop subfolder di dalam class (front, side, dll)
        for subfolder in os.listdir(class_path):
            subfolder_path = os.path.join(class_path, subfolder)

            if not os.path.isdir(subfolder_path):
                continue

            print(f"   > Subfolder: {subfolder}")

            # Buat folder output subfolder
            output_subfolder_path = os.path.join(output_class_path, subfolder)
            os.makedirs(output_subfolder_path, exist_ok=True)

            # Ambil semua video dalam subfolder
            video_files = [
                f for f in os.listdir(subfolder_path)
                if f.lower().endswith(('.mp4', '.avi', '.mov'))
            ]

            # Loop video
            for video_file in tqdm(video_files):
                video_path = os.path.join(subfolder_path, video_file)

                # 1. Load Frames
                frames = load_frames_from_video(video_path)

                if len(frames) < 1:
                    continue

                # 2. Generate Dynamic Image
                try:
                    dyn_image = get_dynamic_image(frames, normalized=True)

                    # 3. Save Result
                    output_filename = os.path.splitext(video_file)[0] + ".jpg"
                    output_path = os.path.join(output_subfolder_path, output_filename)

                    cv2.imwrite(output_path, dyn_image)

                except Exception as e:
                    print(f"Gagal memproses {video_file}: {e}")

    print("\nSUKSES! Semua Dynamic Image telah dibuat.")
    print(f"Silakan cek folder: {output_root}")


if __name__ == '__main__':
    main()